In [ ]:
# Instalando os pacotes

!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install plotly
!pip install scipy
!pip install scikit-learn
!pip install pingouin

In [ ]:
# Importando os pacotes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.cluster.hierarchy as sch
import scipy.stats as stats
from scipy.stats import zscore
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pingouin as pg
import plotly.express as px
import plotly.io as pio
pio.renderers.default='notebook'

In [ ]:
dados_originais = pd.read_excel('Cluster.xlsx')

In [ ]:
dados_originais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4560 entries, 0 to 4559
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NOME_COLABORADOR     4560 non-null   object 
 1   SALARIOS_INTEGRADOS  4560 non-null   float64
 2   MESES_TEMPO_CASA     4560 non-null   int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 107.0+ KB


In [ ]:
dados_cluster = dados_originais.drop(columns=['NOME_COLABORADOR'])

In [ ]:
descritivo_cluster = dados_cluster.describe().T

In [ ]:
# Padronização para quando há dados em escalas diferentes

cluster_pad = dados_cluster.apply(zscore, ddof=1)

In [ ]:
# Cluster Hierárquico Aglomerativo: single linkage + distância cityblock

# Gerando o dendrograma

plt.figure(figsize=(16,8))
dend_sing = sch.linkage(dados_cluster, method = 'single', metric = 'cityblock')
dendrogram_s = sch.dendrogram(dend_sing, color_threshold = 60, labels = list(dados_originais.NOME_COLABORADOR))
plt.title('Dendrograma', fontsize=16)
plt.xlabel('NOME_COLABORADOR', fontsize=16)
plt.ylabel('Distância Cityblock (Manhattan)', fontsize=16)
plt.axhline(y = 60, color = 'red', linestyle = '--')
plt.show()

# Opções para o método de encadeamento ("method"):
    ## single
    ## complete
    ## average

# Opções para as distâncias ("metric"):
    ## euclidean
    ## sqeuclidean
    ## cityblock
    ## chebyshev
    ## canberra
    ## correlation

# Criando a variável que indica os clusters no banco de dados

cluster_sing = AgglomerativeClustering(n_clusters = 3, metric = 'cityblock', linkage = 'single')
indica_cluster_sing = cluster_sing.fit_predict(dados_cluster)
dados_originais['cluster_single'] = indica_cluster_sing
dados_originais['cluster_single'] = dados_originais['cluster_single'].astype('category')

KeyboardInterrupt: 

In [ ]:
#%% Plotando as observações e seus clusters (single + cityblock)

plt.figure(figsize=(10,10))
fig = sns.scatterplot(x='atendimento', y='sortimento', s=60, data=dados_originais, hue='cluster_single')
plt.title('Clusters', fontsize=16)
plt.xlabel('Atendimento', fontsize=16)
plt.ylabel('Sortimento', fontsize=16)
plt.show()

In [ ]:
# Método K-Means Hierárquico

# Considerando que identificamos 3 possíveis clusters na análise hierárquica

kmeans_dados_originais = KMeans(n_clusters=3, init='random', random_state=100).fit(dados_cluster)

# Criando a variável que indica os clusters no banco de dados

kmeans_clusters = kmeans_dados_originais.labels_
dados_originais['cluster_kmeans'] = kmeans_clusters
dados_originais['cluster_kmeans'] = dados_originais['cluster_kmeans'].astype('category')
## O padrão dos clusters é o mesmo dos métodos hierárquicos anteriores

In [ ]:
# Método K-means Não Hierárquico

# Vamos considerar 3 clusters, considerando as evidências anteriores!

kmeans_final = KMeans(n_clusters = 3, init = 'random', random_state=100).fit(cluster_pad)

# Gerando a variável para identificarmos os clusters gerados

kmeans_clusters = kmeans_final.labels_
dados_cluster['cluster_kmeans'] = kmeans_clusters
cluster_pad['cluster_kmeans'] = kmeans_clusters
dados_cluster['cluster_kmeans'] = dados_cluster['cluster_kmeans'].astype('category')
cluster_pad['cluster_kmeans'] = cluster_pad['cluster_kmeans'].astype('category')



In [ ]:
# Método Elbow

elbow = []
K = range(1,8) # parâmetro
for k in K:
    kmeanElbow = KMeans(n_clusters=k, init='random', random_state=100).fit(cluster_pad)
    elbow.append(kmeanElbow.inertia_)

plt.figure(figsize=(16,8))
plt.plot(K, elbow, marker='o')
plt.xlabel('Nº Clusters', fontsize=16)
plt.xticks(range(1,8)) # ajustar range
plt.ylabel('WCSS', fontsize=16)
plt.title('Método de Elbow', fontsize=16)
plt.show()



In [ ]:
# Método silhueta

silhueta = []
I = range(2,8) # Parâmetro
for i in I:
    kmeansSil = KMeans(n_clusters=i, init='random', random_state=100).fit(cluster_pad)
    silhueta.append(silhouette_score(cluster_pad, kmeansSil.labels_))

plt.figure(figsize=(16,8))
plt.plot(range(2, 8), silhueta, color = 'purple', marker='o')
plt.xlabel('Nº Clusters', fontsize=16)
plt.ylabel('Silhueta Média', fontsize=16)
plt.title('Método da Silhueta', fontsize=16)
plt.axvline(x = silhueta.index(max(silhueta))+2, linestyle = 'dotted', color = 'red')
plt.show()


In [ ]:
# Coordenadas dos centroides dos clusters finais

cent_finais = pd.DataFrame(kmeans_dados_originais.cluster_centers_)
cent_finais.columns = dados_cluster.columns
cent_finais.index.name = 'cluster'
cent_finais

# Plotando as observações e seus centroides dos clusters

plt.figure(figsize=(10,10))
sns.scatterplot(x='atendimento', y='sortimento', data=dados_originais, hue='cluster_kmeans', palette='viridis', s=100)
sns.scatterplot(x='atendimento', y='sortimento', data=cent_finais, s=40, c='red', label='Centroides', marker="X")
plt.title('Clusters e centroides', fontsize=16)
plt.xlabel('Atendimento', fontsize=16)
plt.ylabel('Sortimento', fontsize=16)
plt.legend()
plt.show()

In [ ]:
# Análise de variância de um fator (ANOVA)

# Interpretação do output:

## cluster_kmeans MS: indica a variabilidade entre grupos
## Within MS: indica a variabilidade dentro dos grupos
## F: estatística de teste (cluster_kmeans MS / Within MS)
## p-unc: p-valor da estatística F
## se p-valor < 0.05: pelo menos um cluster apresenta média estatisticamente diferente dos demais

pg.anova(dv='SALARIOS_INTEGRADOS',
         between='cluster_kmeans',
         data=cluster_pad,
         detailed=True).T

pg.anova(dv='MESES_TEMPO_CASA',
         between='cluster_kmeans',
         data=cluster_pad,
         detailed=True).T

In [ ]:
#%% Gráfico 3D dos clusters

# Perspectiva 1

fig = px.scatter_3d(dados_cluster,
                    x='SALARIOS_INTEGRADOS',
                    y='MESES_TEMPO_CASA',
                    color='cluster_kmeans')
fig.show()

In [ ]:
# Identificação das características dos clusters

# Agrupando o banco de dados

cluster_grupo = dados_cluster.drop(columns=['NOME_COLABORADOR']).groupby(by=['cluster_kmeans'])

# Estatísticas descritivas por grupo

tab_desc_grupo = cluster_grupo.describe().T